In [7]:
import torch
import torchvision
import torchvision.transforms as T
from torchvision.datasets import VOCSegmentation
from torch.utils.data import DataLoader
import numpy as np

# 1. Dataset Preparation
# FCN-ResNet50 expects images normalized with ImageNet mean & std
image_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

target_transform = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.NEAREST),
    T.CenterCrop(224)
])

def collate_fn(batch):
    images = []
    targets = []
    for img, target in batch:
        images.append(image_transform(img))
        target_tensor = torch.as_tensor(np.array(target_transform(target)), dtype=torch.int64)
        targets.append(target_tensor)
    return torch.stack(images), torch.stack(targets)

print("Downloading PASCAL VOC 2012 Dataset...")
dataset = VOCSegmentation(root='./data', year='2012', image_set='val', download=True)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

# 2. Model Loading
print("Loading pretrained FCN-ResNet50...")
model = torchvision.models.segmentation.fcn_resnet50(weights='DEFAULT')
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Evaluating on device: {device}")

# 3. Evaluation Setup
num_classes = 21  # 20 foreground classes + 1 background class
total_intersection = torch.zeros(num_classes).to(device)
total_union = torch.zeros(num_classes).to(device)

# 4. Evaluation Loop
print("Evaluating (running 5 batches for demonstration)... ")
with torch.no_grad():
    for i, (images, targets) in enumerate(dataloader):
        if i >= 5:  # Limit batches for quick demonstration
            break

        images = images.to(device)
        targets = targets.to(device)

        # Forward pass
        outputs = model(images)['out']
        preds = torch.argmax(outputs, dim=1)

        # Exclude border/void label (255) from loss and metrics calculation
        mask = (targets >= 0) & (targets < 21)
        pred_flat = preds[mask]
        target_flat = targets[mask]

        for cls in range(num_classes):
            pred_cls = (pred_flat == cls)
            target_cls = (target_flat == cls)

            intersection = (pred_cls & target_cls).sum()
            union = (pred_cls | target_cls).sum()

            total_intersection[cls] += intersection
            total_union[cls] += union

# 5. mIoU Calculation
iou_per_class = []
for cls in range(num_classes):
    if total_union[cls] == 0:
        iou_per_class.append(float('nan'))
    else:
        iou = (total_intersection[cls] / total_union[cls]).item()
        iou_per_class.append(iou)
        print(f"Class {cls:2d} IoU: {iou:.4f}")

valid_ious = [val for val in iou_per_class if not np.isnan(val)]
mean_iou = sum(valid_ious) / len(valid_ious) if valid_ious else 0
print(f"\nMean Intersection over Union (mIoU): {mean_iou:.4f}")

100%|██████████| 2.00G/2.00G [00:52<00:00, 37.9MB/s]


Loading pretrained FCN-ResNet50...
Downloading: "https://download.pytorch.org/models/fcn_resnet50_coco-1167a1af.pth" to /root/.cache/torch/hub/checkpoints/fcn_resnet50_coco-1167a1af.pth


100%|██████████| 135M/135M [00:00<00:00, 218MB/s]


Evaluating on device: cpu
Evaluating (running 5 batches for demonstration)... 
Class  0 IoU: 0.9286
Class  1 IoU: 0.6077
Class  2 IoU: 0.6031
Class  4 IoU: 0.4182
Class  5 IoU: 0.9111
Class  6 IoU: 0.9506
Class  7 IoU: 0.0000
Class  9 IoU: 0.0000
Class 10 IoU: 0.9553
Class 11 IoU: 0.0000
Class 13 IoU: 0.8858
Class 15 IoU: 0.9232
Class 16 IoU: 0.0000
Class 17 IoU: 0.9644
Class 18 IoU: 0.7281
Class 19 IoU: 0.9301
Class 20 IoU: 0.9392

Mean Intersection over Union (mIoU): 0.6321


In [1]:
import zipfile
import os

# Unzip the lab4_inlab.zip if it has not been unzipped yet
zip_path = '/content/lab4_inlab.zip'
extract_path = '/content/'

if os.path.exists(zip_path):
    print(f"Extracting {zip_path} to {extract_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete!")
else:
    print(f"{zip_path} not found.")

# List the files in the directory to see the setup
print("\nFiles in /content:")
print(os.listdir('/content'))

Extracting /content/lab4_inlab.zip to /content/...
Extraction complete!

Files in /content:
['.config', 'ELEC475_Lab4_Pre-lab.pdf', 'models', 'lab4_inlab.zip', 'ELEC475_Lab4_In-lab.pdf', 'ELEC475_Lab4_Post-lab.pdf', 'train.py', 'evaluate.py', 'utils', 'test_kd.py']
